# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [ ]:
# YOUR CHANGES HERE

import pandas as pd
transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

# Part 1: policy always chooses "do-nothing"
action = "do-nothing"

# Get all states
states = rewards["state"].unique()

# Initialize values to zero
values = {state: 0.0 for state in states}

# Value iteration settings
tolerance = 1e-10
max_iterations = 10000

for iteration in range(max_iterations):
    new_values = values.copy()
    delta = 0.0

    for state in states:

        # Recovered is terminal
        if state == "recovered":
            new_values[state] = 0.0
            continue

        # Immediate reward for taking do-nothing in this state
        reward = rewards[
            (rewards["action"] == action) &
            (rewards["state"] == state)
        ]["reward"].iloc[0]

        # Possible transitions from this state
        state_transitions = transitions[
            (transitions["action"] == action) &
            (transitions["state"] == state)
        ]

        # Expected future value
        expected_future_value = 0.0

        for _, row in state_transitions.iterrows():
            expected_future_value += (
                row["probability"] * values[row["next_state"]]
            )

        # No discount factor
        new_values[state] = reward + expected_future_value

        # Track largest change
        delta = max(
            delta,
            abs(new_values[state] - values[state])
        )

    values = new_values

    # Stop once values converge
    if delta < tolerance:
        print(f"Converged after {iteration + 1} iterations")
        break


# Convert expected rewards to expected discomfort
discomfort = {
    state: -value
    for state, value in values.items()
}

results = pd.DataFrame({
    "state": states,
    "expected_reward": [values[s] for s in states],
    "expected_discomfort": [discomfort[s] for s in states]
})

print(results)

Converged after 83 iterations
        state  expected_reward  expected_discomfort
0   exposed-1        -3.413333             3.413333
1   exposed-2        -4.266667             4.266667
2   exposed-3        -5.333333             5.333333
3  symptoms-1        -6.666667             6.666667
4  symptoms-2        -5.000000             5.000000
5  symptoms-3        -1.666667             1.666667
6   recovered         0.000000            -0.000000


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [3]:
# YOUR CHANGES HERE

do_nothing_discomfort = pd.DataFrame({
    "state": states,
    "expected_discomfort": [-values[state] for state in states]
})
do_nothing_discomfort.to_csv(
    "do-nothing-discomfort.tsv",
    sep="\t",
    index=False
)
print(do_nothing_discomfort)

        state  expected_discomfort
0   exposed-1             3.413333
1   exposed-2             4.266667
2   exposed-3             5.333333
3  symptoms-1             6.666667
4  symptoms-2             5.000000
5  symptoms-3             1.666667
6   recovered            -0.000000


Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [4]:
# YOUR CHANGES HERE

transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

states = rewards["state"].unique()
actions = rewards["action"].unique()

# Initialize state values
values = {state: 0.0 for state in states}

tolerance = 1e-10
max_iterations = 10000

# Value iteration
for iteration in range(max_iterations):

    new_values = values.copy()
    delta = 0.0

    for state in states:

        # Recovered is terminal
        if state == "recovered":
            new_values[state] = 0.0
            continue

        action_values = {}

        # Evaluate every possible action
        for action in actions:

            # Immediate reward
            reward = rewards[
                (rewards["state"] == state) &
                (rewards["action"] == action)
            ]["reward"].iloc[0]

            # Transitions for this state/action pair
            state_transitions = transitions[
                (transitions["state"] == state) &
                (transitions["action"] == action)
            ]

            # Expected future value
            expected_future_value = 0.0

            for _, row in state_transitions.iterrows():
                expected_future_value += (
                    row["probability"]
                    * values[row["next_state"]]
                )

            # Q(s, a)
            action_values[action] = (
                reward + expected_future_value
            )

        # Optimal action = action with highest reward
        new_values[state] = max(action_values.values())

        delta = max(
            delta,
            abs(new_values[state] - values[state])
        )

    values = new_values

    if delta < tolerance:
        print(f"Converged after {iteration + 1} iterations")
        break

Converged after 100 iterations


Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [6]:
optimal_policy = {}

for state in states:

    if state == "recovered":
        optimal_policy[state] = "do-nothing"
        continue

    action_values = {}

    for action in actions:

        reward = rewards[
            (rewards["state"] == state) &
            (rewards["action"] == action)
        ]["reward"].iloc[0]

        state_transitions = transitions[
            (transitions["state"] == state) &
            (transitions["action"] == action)
        ]

        expected_future_value = sum(
            row["probability"] * values[row["next_state"]]
            for _, row in state_transitions.iterrows()
        )

        action_values[action] = (
            reward + expected_future_value
        )

    # Choose action with highest expected reward
    optimal_policy[state] = max(
        action_values,
        key=action_values.get
    )


# Display treatment plan
treatment_plan = pd.DataFrame({
    "state": states,
    "action": [optimal_policy[state] for state in states]
})

print(treatment_plan)

        state      action
0   exposed-1     sleep-8
1   exposed-2     sleep-8
2   exposed-3     sleep-8
3  symptoms-1    drink-oj
4  symptoms-2    drink-oj
5  symptoms-3    drink-oj
6   recovered  do-nothing


In [8]:
# YOUR CHANGES HERE

minimum_discomfort_actions = pd.DataFrame({
    "state": states,
    "action": [optimal_policy[state] for state in states]
})
minimum_discomfort_actions.to_csv(
    "minimum-discomfort-actions.tsv",
    sep="\t",
    index=False
)
print(minimum_discomfort_actions)

        state      action
0   exposed-1     sleep-8
1   exposed-2     sleep-8
2   exposed-3     sleep-8
3  symptoms-1    drink-oj
4  symptoms-2    drink-oj
5  symptoms-3    drink-oj
6   recovered  do-nothing


Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [9]:
# YOUR CHANGES HERE

# Evaluate the optimal policy from Part 2

policy_values = {state: 0.0 for state in states}

tolerance = 1e-10
max_iterations = 10000

for iteration in range(max_iterations):

    new_values = policy_values.copy()
    delta = 0.0

    for state in states:

        # Recovered is terminal
        if state == "recovered":
            new_values[state] = 0.0
            continue

        # Use the optimal action found in Part 2
        action = optimal_policy[state]

        # Immediate reward
        reward = rewards[
            (rewards["state"] == state) &
            (rewards["action"] == action)
        ]["reward"].iloc[0]

        # Get transitions for the selected action
        state_transitions = transitions[
            (transitions["state"] == state) &
            (transitions["action"] == action)
        ]

        # Expected future value
        expected_future_value = 0.0

        for _, row in state_transitions.iterrows():
            expected_future_value += (
                row["probability"]
                * policy_values[row["next_state"]]
            )

        # Bellman policy evaluation update
        new_values[state] = (
            reward + expected_future_value
        )

        delta = max(
            delta,
            abs(new_values[state] - policy_values[state])
        )

    policy_values = new_values

    if delta < tolerance:
        print(f"Converged after {iteration + 1} iterations")
        break


# Convert rewards to discomfort
optimal_discomfort = pd.DataFrame({
    "state": states,
    "expected_discomfort": [
        -policy_values[state] for state in states
    ]
})

print(optimal_discomfort)

Converged after 100 iterations
        state  expected_discomfort
0   exposed-1                 0.75
1   exposed-2                 1.50
2   exposed-3                 3.00
3  symptoms-1                 6.00
4  symptoms-2                 4.50
5  symptoms-3                 1.50
6   recovered                -0.00


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [11]:
# YOUR CHANGES HERE

optimal_discomfort.to_csv(
    "minimum-discomfort-values.tsv",
    sep="\t",
    index=False
)
print(optimal_discomfort)

        state  expected_discomfort
0   exposed-1                 0.75
1   exposed-2                 1.50
2   exposed-3                 3.00
3  symptoms-1                 6.00
4  symptoms-2                 4.50
5  symptoms-3                 1.50
6   recovered                -0.00


Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [12]:
# YOUR CHANGES HERE

# Create new reward function for minimizing Twizzleflu duration

duration_rewards = rewards.copy()

# -1 if currently symptomatic, 0 otherwise
duration_rewards["reward"] = duration_rewards["state"].apply(
    lambda state: -1.0 if state.startswith("symptoms-") else 0.0
)

print(duration_rewards)

        action       state  reward
0   do-nothing   exposed-1     0.0
1   do-nothing   exposed-2     0.0
2   do-nothing   exposed-3     0.0
3   do-nothing  symptoms-1    -1.0
4   do-nothing  symptoms-2    -1.0
5   do-nothing  symptoms-3    -1.0
6   do-nothing   recovered     0.0
7     drink-oj   exposed-1     0.0
8     drink-oj   exposed-2     0.0
9     drink-oj   exposed-3     0.0
10    drink-oj  symptoms-1    -1.0
11    drink-oj  symptoms-2    -1.0
12    drink-oj  symptoms-3    -1.0
13    drink-oj   recovered     0.0
14     sleep-8   exposed-1     0.0
15     sleep-8   exposed-2     0.0
16     sleep-8   exposed-3     0.0
17     sleep-8  symptoms-1    -1.0
18     sleep-8  symptoms-2    -1.0
19     sleep-8  symptoms-3    -1.0
20     sleep-8   recovered     0.0


In [13]:
# Initialize values
duration_values = {state: 0.0 for state in states}

tolerance = 1e-10
max_iterations = 10000

for iteration in range(max_iterations):

    new_values = duration_values.copy()
    delta = 0.0

    for state in states:

        if state == "recovered":
            new_values[state] = 0.0
            continue

        action_values = {}

        for action in actions:

            # New duration-based reward
            reward = duration_rewards[
                (duration_rewards["state"] == state) &
                (duration_rewards["action"] == action)
            ]["reward"].iloc[0]

            # Original transition probabilities stay unchanged
            state_transitions = transitions[
                (transitions["state"] == state) &
                (transitions["action"] == action)
            ]

            expected_future_value = sum(
                row["probability"]
                * duration_values[row["next_state"]]
                for _, row in state_transitions.iterrows()
            )

            action_values[action] = (
                reward + expected_future_value
            )

        # Highest reward = shortest expected duration
        new_values[state] = max(action_values.values())

        delta = max(
            delta,
            abs(new_values[state] - duration_values[state])
        )

    duration_values = new_values

    if delta < tolerance:
        print(f"Converged after {iteration + 1} iterations")
        break

Converged after 84 iterations


In [14]:
duration_policy = {}

for state in states:

    if state == "recovered":
        duration_policy[state] = "do-nothing"
        continue

    action_values = {}

    for action in actions:

        reward = duration_rewards[
            (duration_rewards["state"] == state) &
            (duration_rewards["action"] == action)
        ]["reward"].iloc[0]

        state_transitions = transitions[
            (transitions["state"] == state) &
            (transitions["action"] == action)
        ]

        expected_future_value = sum(
            row["probability"]
            * duration_values[row["next_state"]]
            for _, row in state_transitions.iterrows()
        )

        action_values[action] = (
            reward + expected_future_value
        )

    duration_policy[state] = max(
        action_values,
        key=action_values.get
    )


duration_plan = pd.DataFrame({
    "state": states,
    "action": [duration_policy[state] for state in states]
})

print(duration_plan)

        state      action
0   exposed-1     sleep-8
1   exposed-2     sleep-8
2   exposed-3     sleep-8
3  symptoms-1  do-nothing
4  symptoms-2  do-nothing
5  symptoms-3  do-nothing
6   recovered  do-nothing


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [15]:
# YOUR CHANGES HERE

duration_rewards.to_csv(
    "duration-rewards.tsv",
    sep="\t",
    index=False
)
print(duration_rewards)

        action       state  reward
0   do-nothing   exposed-1     0.0
1   do-nothing   exposed-2     0.0
2   do-nothing   exposed-3     0.0
3   do-nothing  symptoms-1    -1.0
4   do-nothing  symptoms-2    -1.0
5   do-nothing  symptoms-3    -1.0
6   do-nothing   recovered     0.0
7     drink-oj   exposed-1     0.0
8     drink-oj   exposed-2     0.0
9     drink-oj   exposed-3     0.0
10    drink-oj  symptoms-1    -1.0
11    drink-oj  symptoms-2    -1.0
12    drink-oj  symptoms-3    -1.0
13    drink-oj   recovered     0.0
14     sleep-8   exposed-1     0.0
15     sleep-8   exposed-2     0.0
16     sleep-8   exposed-3     0.0
17     sleep-8  symptoms-1    -1.0
18     sleep-8  symptoms-2    -1.0
19     sleep-8  symptoms-3    -1.0
20     sleep-8   recovered     0.0


Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [16]:
# YOUR CHANGES HERE


duration_values = {state: 0.0 for state in states}

tolerance = 1e-10
max_iterations = 10000

for iteration in range(max_iterations):

    new_values = duration_values.copy()
    delta = 0.0

    for state in states:

        # Recovered is terminal
        if state == "recovered":
            new_values[state] = 0.0
            continue

        action_values = {}

        # Try every action
        for action in actions:

            # Reward from our Part 4 reward function
            reward = duration_rewards[
                (duration_rewards["state"] == state) &
                (duration_rewards["action"] == action)
            ]["reward"].iloc[0]

            # Original transition probabilities
            state_transitions = transitions[
                (transitions["state"] == state) &
                (transitions["action"] == action)
            ]

            expected_future_value = sum(
                row["probability"] *
                duration_values[row["next_state"]]
                for _, row in state_transitions.iterrows()
            )

            # Q(s,a)
            action_values[action] = (
                reward + expected_future_value
            )

        # Choose highest reward
        new_values[state] = max(action_values.values())

        delta = max(
            delta,
            abs(new_values[state] - duration_values[state])
        )

    duration_values = new_values

    if delta < tolerance:
        print(f"Converged after {iteration + 1} iterations")
        break

Converged after 84 iterations


In [18]:
duration_policy = {}

for state in states:

    if state == "recovered":
        duration_policy[state] = "do-nothing"
        continue

    action_values = {}

    for action in actions:

        reward = duration_rewards[
            (duration_rewards["state"] == state) &
            (duration_rewards["action"] == action)
        ]["reward"].iloc[0]

        state_transitions = transitions[
            (transitions["state"] == state) &
            (transitions["action"] == action)
        ]

        expected_future_value = sum(
            row["probability"] *
            duration_values[row["next_state"]]
            for _, row in state_transitions.iterrows()
        )

        action_values[action] = (
            reward + expected_future_value
        )

    duration_policy[state] = max(
        action_values,
        key=action_values.get
    )

duration_plan = pd.DataFrame({
    "state": states,
    "action": [duration_policy[state] for state in states]
})

print(duration_plan)

        state      action
0   exposed-1     sleep-8
1   exposed-2     sleep-8
2   exposed-3     sleep-8
3  symptoms-1  do-nothing
4  symptoms-2  do-nothing
5  symptoms-3  do-nothing
6   recovered  do-nothing


Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [19]:
# YOUR CHANGES HERE

minimum_duration_actions = pd.DataFrame({
    "state": states,
    "action": [duration_policy[state] for state in states]
})

minimum_duration_actions.to_csv(
    "minimum-duration-actions.tsv",
    sep="\t",
    index=False
)
print(minimum_duration_actions)

        state      action
0   exposed-1     sleep-8
1   exposed-2     sleep-8
2   exposed-3     sleep-8
3  symptoms-1  do-nothing
4  symptoms-2  do-nothing
5  symptoms-3  do-nothing
6   recovered  do-nothing


Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [20]:
# YOUR CHANGES HERE

sick_values = {state: 0.0 for state in states}

tolerance = 1e-10
max_iterations = 10000

for iteration in range(max_iterations):

    new_values = sick_values.copy()
    delta = 0.0

    for state in states:

        # Recovered is terminal
        if state == "recovered":
            new_values[state] = 0.0
            continue

        # Use optimal duration action from Part 5
        action = duration_policy[state]

        # Duration reward from Part 4
        reward = duration_rewards[
            (duration_rewards["state"] == state) &
            (duration_rewards["action"] == action)
        ]["reward"].iloc[0]

        # Transitions for selected action
        state_transitions = transitions[
            (transitions["state"] == state) &
            (transitions["action"] == action)
        ]

        expected_future_value = sum(
            row["probability"] * sick_values[row["next_state"]]
            for _, row in state_transitions.iterrows()
        )

        new_values[state] = reward + expected_future_value

        delta = max(
            delta,
            abs(new_values[state] - sick_values[state])
        )

    sick_values = new_values

    if delta < tolerance:
        print(f"Converged after {iteration + 1} iterations")
        break


# Convert negative rewards to expected days sick
expected_days_sick = pd.DataFrame({
    "state": states,
    "expected_days_sick": [
        -sick_values[state] for state in states
    ]
})

print(expected_days_sick)

Converged after 84 iterations
        state  expected_days_sick
0   exposed-1            1.250000
1   exposed-2            2.500000
2   exposed-3            5.000000
3  symptoms-1           10.000000
4  symptoms-2            6.666667
5  symptoms-3            3.333333
6   recovered           -0.000000


Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [21]:
# YOUR CHANGES HERE

minimum_duration_days = pd.DataFrame({
    "state": states,
    "expected_sick_days": [
        -sick_values[state] for state in states
    ]
})

minimum_duration_days.to_csv(
    "minimum-duration-days.tsv",
    sep="\t",
    index=False
)
print(minimum_duration_days)

        state  expected_sick_days
0   exposed-1            1.250000
1   exposed-2            2.500000
2   exposed-3            5.000000
3  symptoms-1           10.000000
4  symptoms-2            6.666667
5  symptoms-3            3.333333
6   recovered           -0.000000


Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

Hint: Make sure that you are reporting on expected discomfort, not repeating the expected number of days.

In [22]:
# YOUR CHANGES HERE

# Part 7: Expected discomfort under the minimum-duration policy

speed_discomfort_values = {state: 0.0 for state in states}

tolerance = 1e-10
max_iterations = 10000

for iteration in range(max_iterations):

    new_values = speed_discomfort_values.copy()
    delta = 0.0

    for state in states:

        # Recovered is terminal
        if state == "recovered":
            new_values[state] = 0.0
            continue

        # Use the policy that minimizes days sick
        action = duration_policy[state]

        # IMPORTANT: use ORIGINAL discomfort rewards
        reward = rewards[
            (rewards["state"] == state) &
            (rewards["action"] == action)
        ]["reward"].iloc[0]

        # Transition probabilities for that action
        state_transitions = transitions[
            (transitions["state"] == state) &
            (transitions["action"] == action)
        ]

        expected_future_value = sum(
            row["probability"] *
            speed_discomfort_values[row["next_state"]]
            for _, row in state_transitions.iterrows()
        )

        new_values[state] = (
            reward + expected_future_value
        )

        delta = max(
            delta,
            abs(
                new_values[state]
                - speed_discomfort_values[state]
            )
        )

    speed_discomfort_values = new_values

    if delta < tolerance:
        print(f"Converged after {iteration + 1} iterations")
        break

Converged after 82 iterations


In [23]:
speed_discomfort = pd.DataFrame({
    "state": states,
    "expected_discomfort": [
        -speed_discomfort_values[state]
        for state in states
    ]
})

print(speed_discomfort)

        state  expected_discomfort
0   exposed-1             0.833333
1   exposed-2             1.666667
2   exposed-3             3.333333
3  symptoms-1             6.666667
4  symptoms-2             5.000000
5  symptoms-3             1.666667
6   recovered            -0.000000


In [24]:
comparison = pd.DataFrame({
    "state": states,

    "minimum_discomfort_policy": [
        -policy_values[state]
        for state in states
    ],

    "minimum_duration_policy": [
        -speed_discomfort_values[state]
        for state in states
    ]
})

comparison["extra_discomfort_for_speed"] = (
    comparison["minimum_duration_policy"]
    - comparison["minimum_discomfort_policy"]
)

print(comparison)

        state  minimum_discomfort_policy  minimum_duration_policy  \
0   exposed-1                       0.75                 0.833333   
1   exposed-2                       1.50                 1.666667   
2   exposed-3                       3.00                 3.333333   
3  symptoms-1                       6.00                 6.666667   
4  symptoms-2                       4.50                 5.000000   
5  symptoms-3                       1.50                 1.666667   
6   recovered                      -0.00                -0.000000   

   extra_discomfort_for_speed  
0                    0.083333  
1                    0.166667  
2                    0.333333  
3                    0.666667  
4                    0.500000  
5                    0.166667  
6                    0.000000  


Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [25]:
# YOUR CHANGES HERE

policy_comparison = pd.DataFrame({
    "state": states,
    "speed_discomfort": [
        -speed_discomfort_values[state]
        for state in states
    ],
    "minimize_discomfort": [
        -policy_values[state]
        for state in states
    ]
})

policy_comparison.to_csv(
    "policy-comparison.tsv",
    sep="\t",
    index=False
)
print(policy_comparison)

        state  speed_discomfort  minimize_discomfort
0   exposed-1          0.833333                 0.75
1   exposed-2          1.666667                 1.50
2   exposed-3          3.333333                 3.00
3  symptoms-1          6.666667                 6.00
4  symptoms-2          5.000000                 4.50
5  symptoms-3          1.666667                 1.50
6   recovered         -0.000000                -0.00


Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.